In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path
project_root = Path.cwd()
if not (project_root / "utils").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

True

In [23]:
from tqdm.auto import tqdm
from utils.seku_crd_corpus_utils import *
from prediction.predict_ner import *
from prediction.predict_ner_v2 import *
from prediction.predict_narrative import *
from utils.embedding_utils import *
import traceback
import random

In [4]:
LOCAL_MODEL_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/model"
WANDB_PROJECT = "psm-iiser-citation-relevancy"
WANDB_ARTIFACT = "qwen-ner-adapter"
MAX_REFERENCES = 32 # Prediction is in GPU, to quickly test take only few references
RESULT_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/dataset-ner-synthetic-dgft-6000/results"
TEST_SET_MAIN_PAPERS = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/main-paper-set-1.txt"
CITATION_CONTEXT_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/citation_contexts"

**Load Main papers from Paper Index**

In [5]:
paper_index = load_paper_index(max_files=4096) # turn off the max-files if you want to load all papers from dataset
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 4095line [00:00, 4294.68line/s]


Loaded 4096 papers from /Work/IISER/citation-data/SEKU-CRD-v1/paper_index_2024.jsonl (201.8 MB in memory)
Total 4096 Main papers


**Take some samples of main papers**

In [6]:
sample = 50
sample_main_paper_ids = random.sample(all_main_paper_ids, sample)
print(f'Total {len(sample_main_paper_ids)} Main papers')

Total 50 Main papers


### Main Processing Block **[Heart of SEKU]** ###

**Model Loading**

In [ ]:
ner_model = FineTunedNERModel(local_dir=LOCAL_MODEL_DIR,wandb_project=WANDB_PROJECT,artifact_name=WANDB_ARTIFACT) # for ner
embedding_model = EmbeddingModel()
embedding_model.load_embedding_model()

In [6]:
# a very useful function to help on the way
def get_entities_from_paper(abstract,ner_model,max_new_tokens=2048):
    result = ner_model.predict(abstract,max_new_tokens=max_new_tokens)
    result_g = combine_entities_by_label(result) # raw entities only
    result_aug = augment_context_by_label(result,abstract,window=0) # grouped entities + context 
    all_windows = {}
    for r in result_aug:
        text = ""
        for rt in result_aug[r]:
            text += rt["text"]
        all_windows[r] = text
    return result_g,result_aug,all_windows


# same as get_entities_from_paper, but for a list of {paper_id,text} at once (no result_g)
# uses the class-based model's real batched generation (apredict) instead of one generate() call per paper
async def get_entities_from_papers(items,ner_model,batch_size=8,max_new_tokens=2048):
    text_by_id = {item["paper_id"]:item["text"] for item in items}
    ft_items = [{"paper_id":item["paper_id"],"text_to_extract":item["text"]} for item in items]
    predictions = await ner_model.apredict(ft_items,batch_size=batch_size,max_new_tokens=max_new_tokens)

    result_aug_map = {}
    all_windows_map = {}
    for pred in predictions:
        paper_id = pred["paper_id"]
        entities = pred["entities"]
        if not entities: # apredict returns {} for paper_ids that failed to parse
            result_aug_map[paper_id] = {}
            all_windows_map[paper_id] = {}
            continue
        text = text_by_id[paper_id]
        result_aug = augment_context_by_label(entities,text,window=0) # grouped entities + context
        all_windows = {}
        for r in result_aug:
            w_text = ""
            for rt in result_aug[r]:
                w_text += rt["text"]
            all_windows[r] = w_text
        result_aug_map[paper_id] = result_aug
        all_windows_map[paper_id] = all_windows
    return result_aug_map,all_windows_map

#### **Quick Test Extraction [WIP]** ####

In [ ]:
test_abstract = """
We solve the equations of motion of five-dimensional ungauged supergravity\ncoupled to three U(1) gauge fields using a floating-brane Ansatz in which the\nelectric potentials are directly related to the gravitational warp factors. We\nfind a new class of non-BPS solutions, that can be obtained linearly starting\nfrom an Euclidean four-dimensional Einstein-Maxwell base. This class - the\nlargest known so far - reduces to the BPS and almost-BPS solutions in certain\nlimits. We solve the equations explicitly when the base space is given by the\nIsrael-Wilson metric, and obtain solutions describing non-BPS D6 and anti-D6\nbranes kept in equilibrium by flux. We also examine the action of spectral flow\non solutions with an Israel-Wilson base and show that it relates these\nsolutions to almost-BPS solutions with a Gibbons-Hawking base.
"""
result_g,result_aug,all_windows = get_entities_from_paper(test_abstract,ner_model,max_new_tokens=2048)

In [28]:
for ne in all_windows:
    print(ne)
    print("=="*5)
    print(all_windows[ne])

METHOD
 We solve the <equations of motion of five-dimensional ungauged supergravity> coupled to three U(1) gauge fields using a <floating-brane Ansatz> in which the electric potentials are directly related to the gravitational warp factors.We <solve the equations explicitly when the base space is given by the Israel-Wilson metric>, and obtain solutions describing non-BPS D6 and anti-D6 branes kept in equilibrium by flux.We also examine the action of <spectral flow> on solutions with an Israel-Wilson base and show that it relates these solutions to almost-BPS solutions with a Gibbons-Hawking base.
KEY_CONCEPT
 We solve the equations of motion of five-dimensional ungauged supergravity coupled to <three U(1) gauge fields> using a floating-brane Ansatz in which the electric potentials are directly related to the <gravitational warp factors>.We find a new class of non-BPS solutions, that can be obtained linearly starting from an <Euclidean four-dimensional Einstein-Maxwell base>.This class 

**Randomly pick a main paper from samples**

In [47]:
rand_idx = random.randint(0,sample-1)
rand_paper_id = sample_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
print(f'Title: {title}')
main_paper_data = get_paper_from_s3(rand_paper_id)
print("Retrieved main paper data from S3...Extracting titles (not needed) and abstract")
main_title,main_abstract = get_title_and_abstract(main_paper_data)
print("Extracted title and abstract from raw data...Finding Named Entities")

Random Main paper Index:39 [Corpus ID: 2009.11817]
Title: The modified logarithmic Sobolev inequality for quantum spin systems:
  classical and commuting nearest neighbour interactions
Retrieved main paper data from S3...Extracting titles (not needed) and abstract
Extracted title and abstract from raw data...Finding Named Entities


**Extract Entities from Main Paper**

In [48]:
abstracts = [{"paper_id":rand_paper_id,"text":main_abstract}]
result_aug_map,all_windows_map = await get_entities_from_papers(abstracts,ner_model)
print("Entity Extraction Findinings complete")

Extracting entities: 100%|██████████| 1/1 [00:21<00:00, 21.87s/paper]

Entity Extraction Findinings complete


In [36]:
print(main_abstract) # just check abstract for reference

Recent progress in Higgs boson studies for the International e+e- Linear
Collider (ILC) is reported. These studies include extended simulations of the
measurement of the Higgs mass, measurements of the Higgs boson branching ratios
at higher center-of-mass energies, and methods for extracting the Higgs boson
self-coupling. Also, the interplay between the LHC and the ILC in the
measurement of the top Yukawa coupling and in the extraction of the
supersymmetric Higgs sector parameters is discussed.


**See Entities in Context by Label**

In [49]:
for paper_id in all_windows_map:
    print(paper_id)
    print("="*5)
    data = all_windows_map[paper_id]
    for d in data:
        print(d)
        print("--"*5)
        print(data[d])
        print("--"*5)

2009.11817
=====
METHOD
----------
Given a <uniform, frustration-free family of local Lindbladians> defined on a quantum lattice spin system in any spatial dimension, we prove a strong exponential convergence in relative entropy of the system to equilibrium under a condition of spatial mixing of the stationary Gibbs states and the rapid decay of the relative entropy on finite-size blocks.Moreover, we show that our notion of spatial mixing is a consequence of the recent quantum generalization of <Dobrushin and Shlosman's complete analyticity of the free-energy at equilibrium>.
----------
KEY_CONCEPT
----------
Given a uniform, frustration-free family of local Lindbladians defined on a <quantum lattice spin system> in any spatial dimension, we prove a strong exponential convergence in relative entropy of the system to equilibrium under a condition of spatial mixing of the <stationary Gibbs states> and the rapid decay of the relative entropy on finite-size blocks.Our result leads to the f

#### **Extract Entities Refs [only abstract] [Batch]** ####

In [50]:
in_corpus_refs_ids = main_data["in_corpus_references"] # list of IDs of [C]
nb_all_refs_r = main_data["total_bib_entries"]
# we need to check if these ids truly exists in our Full Paper Data
# our full paper data is in S3 😊, so we will get it from there. 

# for quick computation, we will take few references, not ALL
in_corpus_refs_ids = in_corpus_refs_ids[:min(len(in_corpus_refs_ids),MAX_REFERENCES)]
print(f"We will extract entities from {len(in_corpus_refs_ids)} citations")
results = await get_papers_from_s3(in_corpus_refs_ids)
found_full_data = 0

print(f"Checking refernces for Main paper {rand_paper_id}\n{main_title}")
print("="*5)
# ref_ne_contexts = {}
list_of_texts = []
references_nes = {}
for paper_id, data in results.items():
    if data is not None:
        ref_title,ref_abstract = get_title_and_abstract(data)
        list_of_texts.append({"paper_id":paper_id,"text":ref_abstract})
        references_nes[paper_id] = {"abstract":ref_abstract,"title":ref_title}
        found_full_data += 1
    else:
        print(f"{paper_id} Not found ☹️")
full_data_ratio = found_full_data / nb_all_refs_r
print(f"{full_data_ratio*100:0.2f}% references have full data") 
print(f"Collected {len(list_of_texts)} for NE extraction")

We will extract entities from 32 citations


Fetching papers: 100%|██████████| 32/32 [00:07<00:00,  4.56it/s]

Checking refernces for Main paper 2009.11817
The modified logarithmic Sobolev inequality for quantum spin systems:
  classical and commuting nearest neighbour interactions
=====
27.83% references have full data
Collected 32 for NE extraction


In [51]:
ref_result_aug_map,ref_all_windows_map = await get_entities_from_papers(list_of_texts,ner_model,batch_size=6,max_new_tokens=2048)
# build a dict for next steps
for paper_id in ref_all_windows_map:
    data = ref_all_windows_map[paper_id]
    references_nes[paper_id].update({"nes":data})

Extracting entities: 100%|██████████| 32/32 [04:13<00:00,  7.92s/paper]


**Match Main paper NEs with Ref paper NEs**

In [52]:
all_pairs = {}
for paper_id in all_windows_map: # main paper data
    print(paper_id)
    print("="*5)
    main_nes = all_windows_map[paper_id] # main paper entities
    for d in main_nes:
        d_dict = {}
        for ref_id in references_nes: # for each ref paper
            if "nes" in references_nes[ref_id]:
                ref_nes = references_nes[ref_id]["nes"] # this is a dict of entity labels
                if d in ref_nes:
                    d_dict[ref_id] = ref_nes[d]
        all_pairs[d] = {"base_text":main_nes[d],"ref_texts":d_dict}
print("All pairs collected")

2009.11817
=====
All pairs collected


In [53]:
all_pairs

{'METHOD': {'base_text': "Given a <uniform, frustration-free family of local Lindbladians> defined on a quantum lattice spin system in any spatial dimension, we prove a strong exponential convergence in relative entropy of the system to equilibrium under a condition of spatial mixing of the stationary Gibbs states and the rapid decay of the relative entropy on finite-size blocks.Moreover, we show that our notion of spatial mixing is a consequence of the recent quantum generalization of <Dobrushin and Shlosman's complete analyticity of the free-energy at equilibrium>.",
  'ref_texts': {'1908.09004': 'The mixing time of <Markovian dissipative evolutions> of open quantum many-body systems can be bounded using optimal constants of certain <quantum functional inequalities>, such as the modified logarithmic Sobolev constant.For classical spin systems, the positivity of such constants follows from a mixing condition for the Gibbs measure, via <quasi-factorization results for the entropy>.Insp

In [54]:
report = {}
for lbl in all_pairs:
    base_text = all_pairs[lbl]["base_text"]
    ref_texts = all_pairs[lbl]["ref_texts"]
    rel_score_lbl = reference_relevance_computation(base_text,ref_texts,embedding_model)
    # print(lbl)
    # print("=="*5)
    for pid in rel_score_lbl:
        # print(f"paper {pid} relevance {rel_score_lbl[pid]:0.2f}")
        if pid not in report:
            report[pid] = {lbl:rel_score_lbl[pid]}
        else:
            report[pid].update({lbl:rel_score_lbl[pid]})
# showing the report
for ref_id in report:
    print(ref_id)
    print("--"*5)
    for lbl in report[ref_id]:
        print(f"{lbl}: {report[ref_id][lbl]}")
    print("=="*5)

Embedding: 100%|██████████| 1/1 [00:00<00:00, 132.58text/s]

1908.09004
----------
METHOD: 0.925576388835907
KEY_CONCEPT: 0.946906328201294
METRIC: 0.933276355266571
FINDING: 0.939850389957428
1710.01039
----------
METHOD: 0.8835877776145935
KEY_CONCEPT: 0.916726291179657
METRIC: 0.8832255601882935
CONTRIBUTION: 0.8973941802978516
PROBLEM: 0.8928980827331543
1804.10100
----------
METHOD: 0.8679274320602417
KEY_CONCEPT: 0.8994246125221252
CONTRIBUTION: 0.8642253875732422
1904.11043
----------
METHOD: 0.8945624232292175
KEY_CONCEPT: 0.9247347712516785
METRIC: 0.8839508295059204
CONTRIBUTION: 0.8856003880500793
PROBLEM: 0.8329445123672485
1803.05379
----------
METHOD: 0.8379286527633667
KEY_CONCEPT: 0.9126996994018555
METRIC: 0.9197006821632385
1509.02610
----------
METHOD: 0.8661872744560242
KEY_CONCEPT: 0.8386030793190002
METRIC: 0.825564980506897
FINDING: 0.8432974815368652
1304.6890
----------
METHOD: 0.9171549677848816
KEY_CONCEPT: 0.8808730840682983
METRIC: 0.7901989221572876
FINDING: 0.8890429139137268
0803.1482
----------
METHOD: 0.89256262

**Create Extended Report to save result**

In [56]:
extended_report = {}
for paper_id in all_windows_map: # main paper data
    main_nes = all_windows_map[paper_id] # main paper entities
    extended_report[paper_id] = {"title":main_title,"abstract":main_abstract,"nes":main_nes} # add record for main paper
ref_records = {}
for ref_id in references_nes:
    # ref_title = references_nes[ref_id]["title"]
    ref_nes = references_nes[ref_id]["nes"] # this is a dict of entity labels
    ref_records[ref_id] = {"nes":ref_nes,
                           "title":references_nes[ref_id]["title"],
                           "abstract":references_nes[ref_id]["abstract"],
                           "relevance_score":report[ref_id]}
extended_report[paper_id]["references"] = ref_records

with open(f"{RESULT_DIR}/{paper_id}.json","w") as f:
    json.dump(extended_report,f,indent=3)
print(f"Record created for {paper_id}")
    

Record created for 2009.11817


### Narrative Model Prediction ###

**Load Model**

In [7]:
local_dir = os.environ.get("NER_MODEL_LOCAL_DIR", str(project_root / "model"))
narrative_model = NarrativeModel(local_dir=local_dir,checkpoint_kind="train-best",wandb_version="latest",version="v2")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/parthosarothi/.netrc.


Looking up 'qwen-narrative-adapter-train-best:latest' in W&B project 'psm-iiser-citation-relevancy'...

[Download] Downloading from S3...
  Bucket: psm-iiser
  Prefix: narrative-model/qwen-narrative/train_best
  Downloading: narrative-model/qwen-narrative/train_best/adapter_model/README.md -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/README.md
  Downloading: narrative-model/qwen-narrative/train_best/adapter_model/adapter_config.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/adapter_config.json
  Downloading: narrative-model/qwen-narrative/train_best/adapter_model/adapter_model.safetensors -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/adapter_model.safetensors
  Downloading: narrative-model/qwen-narrative/train_best/adapter_model/added_tokens.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/adapter_model/added_tokens.json
  Downloading: nar

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 154.49it/s]


**Randomly pick a main paper from samples**

In [12]:
rand_idx = random.randint(0,sample-1)
rand_paper_id = sample_main_paper_ids[rand_idx]
print(f"Random Main paper Index:{rand_idx} [Corpus ID: {rand_paper_id}]")
main_data = paper_index[rand_paper_id] # data from paper index file
title = main_data["title"]
print(f'Title: {title}')
main_paper_data = get_paper_from_s3(rand_paper_id)
print("Retrieved main paper data from S3...Extracting titles (not needed) and abstract")
main_title,main_abstract = get_title_and_abstract(main_paper_data)
print("Extracted title and abstract from raw data...Finding Named Entities")

Random Main paper Index:23 [Corpus ID: 2310.02991]
Title: Diagnostic Tomography of Applied Holography
Retrieved main paper data from S3...Extracting titles (not needed) and abstract
Extracted title and abstract from raw data...Finding Named Entities


In [13]:
main_abstract

"The single-particle behavior in $d\\geq 1$-dimensional Fermi gases with a\nlarge number $N$ of species and strong short-range $s$-wave scattering is\ndiscussed in the $2d$ 'tomographic' framework of a (pseudo)holographic\ncorrespondence with a certain $3d$ gravity of the $AdS_3$ type. However, due to\nthe intrinsically topological nature of such a bulk theory its dynamics reduces\nto a purely boundary one and so, akin to its $SYK/AdS_2$ counterpart, this\nformal correspondence neither represents a genuine case of, nor endorses the\nhypothetical generalized holographic duality."

**Quick Test Narrative Prediction**

In [8]:
narrative = narrative_model.predict(main_abstract)
print(json.dumps(narrative, indent=2))

NameError: name 'main_abstract' is not defined

In [17]:
import traceback as tr
async def process_narrative_main_paper(paper_id,paper_index):
    data = {}
    status = False
    main_data = paper_index[paper_id] # data from paper index file
    try:
        main_paper_data = get_paper_from_s3(paper_id)
        main_title,main_abstract = get_title_and_abstract(main_paper_data)
        main_narrative = narrative_model.predict(main_abstract)
        data[paper_id] = {"title":main_title,"abstract":main_abstract,"narrative":main_narrative}
        # now check the references
        in_corpus_refs_ids = main_data["in_corpus_references"]
        in_corpus_refs_ids = in_corpus_refs_ids[:min(len(in_corpus_refs_ids),MAX_REFERENCES)]
        results = await get_papers_from_s3(in_corpus_refs_ids)
        ref_records = {}
        for ref_paper_id, ref_data in results.items():
            if ref_data is not None:
                try:
                    ref_title,ref_abstract = get_title_and_abstract(ref_data)
                    ref_narrative = narrative_model.predict(ref_abstract)
                    ref_records[ref_paper_id] = {"title":ref_title,"abstract":ref_abstract,"narrative":ref_narrative}
                except:
                    print(f"can not process {paper_id}, issue {tr.format_exc()}")
                    pass
            else:
                print(f"Reference {ref_paper_id} Not found ☹️")
        data[paper_id].update({"references":ref_records})
        status = True
    except:
        print(f"can not process {paper_id}, issue {tr.format_exc()}")
    return status,data


In [16]:
status, narrative = await process_narrative_main_paper(rand_paper_id,paper_index)
print(f"status: {status}")

Fetching papers: 100%|██████████| 25/25 [00:01<00:00, 14.55it/s]


status: True


In [17]:
with open("temp.json","w") as f:
    json.dump(narrative,f,indent=2)

**Bulk Run**

In [18]:
with open(TEST_SET_MAIN_PAPERS) as f:
    test_set_main_paper_ids = [line.strip() for line in f if line.strip()]
print(f"Loaded {len(test_set_main_paper_ids)} main paper ids for bulk run")

Loaded 59 main paper ids for bulk run


In [21]:
# check if we already processed it
extracted = set()
for root,_,fnames in os.walk(CITATION_CONTEXT_DIR):
    for fname in fnames:
        mpid = fname.replace(".json","")
        extracted.add(mpid)
print(f"Total {len(extracted)} papers already processed")

Total 56 papers already processed


In [24]:
count = len(extracted)

for t_main_paper_id in test_set_main_paper_ids:
    try:
        if t_main_paper_id in extracted:
            print(f"Main paper {t_main_paper_id} already processed, passing on")
            status = True
            continue
        status, narrative = await process_narrative_main_paper(t_main_paper_id,paper_index)
        if status:
            with open(os.path.join(CITATION_CONTEXT_DIR,f"{t_main_paper_id}.json"),"w") as f:
                json.dump(narrative,f,indent=2)
    except Exception as e:
        print(f"Error {t_main_paper_id} for {tr.format_exc()}")
        pass
    count += 1
    print(f"{t_main_paper_id} - status: {status} - Total {count}/{len(test_set_main_paper_ids)}")

Main paper 0807.1687 already processed, passing on
Main paper 0807.2608 already processed, passing on
Main paper 0807.3039 already processed, passing on
Main paper 1608.08501 already processed, passing on
Main paper 1609.04550 already processed, passing on
Error 1612.05995 for Traceback (most recent call last):
  File "/tmp/ipykernel_107404/2374458977.py", line 9, in <module>
    status, narrative = await process_narrative_main_paper(t_main_paper_id,paper_index)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_107404/2228500970.py", line 5, in process_narrative_main_paper
    main_data = paper_index[paper_id] # data from paper index file
                ~~~~~~~~~~~^^^^^^^^^^
KeyError: '1612.05995'

1612.05995 - status: True - Total 57/59
Main paper 1906.08472 already processed, passing on
Main paper 1908.09797 already processed, passing on
Main paper 1908.09929 already processed, passing on
Main paper 1909.10382 already proc

### Citation Context Explorer ###

In [29]:
from utils.citation_context import *

In [30]:
citation_ctx_extractor = CitationContextExtractor()

In [31]:
for root,_,fnames in os.walk(CITATION_CONTEXT_DIR):
    for fname in fnames:
        with open(os.path.join(root,fname)) as f:
            data = json.load(f)
        # this is a dict with a single key i.e. main paper id
        main_paper_id = list(data.keys())[0]
        status, cite_ctx = find_citation_context(main_paper_id=main_paper_id,
                                         bulk_paper_index=paper_index,
                                         citation_context_extractor=citation_ctx_extractor)
        if status:
            refs = cite_ctx[main_paper_id]["references"] # this is a dict where each key is a ref paper id
            for ref_paper_id in refs:
                ctx = refs[ref_paper_id]["citation_context"]
                data[main_paper_id]["references"][ref_paper_id].update({"citation_context":ctx})
        
        # write back
        with open(os.path.join(root,fname),"w") as f:
            json.dump(data,f,indent=2)
        # break


Extracted context for 2309.07974 success [32/32]
Extracted context for 2312.12669 success [30/30]
Extracted context for 2312.12691 success [16/16]
Extracted context for 0807.1687 success [17/17]
Extracted context for 2303.07736 success [16/16]
Extracted context for 2310.17596 success [32/32]
Extracted context for 2312.08465 success [32/32]
Extracted context for 1908.09929 success [31/32]
Extracted context for 2306.14383 success [32/32]
Extracted context for 2306.03857 success [30/30]
Extracted context for 1609.04550 success [19/19]
Extracted context for 1906.08472 success [26/26]
Extracted context for 2310.02958 success [10/10]
Extracted context for 2304.14073 success [15/15]
Extracted context for 0807.3039 success [21/21]
Extracted context for 2302.03584 success [32/32]
Extracted context for 2307.04602 success [17/17]
Extracted context for 2310.03690 success [6/6]
Extracted context for 2308.00556 success [17/17]
Extracted context for 2308.09757 success [32/32]
Extracted context for 23